# Predicting DoorDash Delivery Duration: Models 3 and 4

## ADS 505 Final Team Project

**Caly Nguyen: Random Forest and Histogram Gradient Boosting**

This notebook uses the prepared training, validation, and test files created in
the team's EDA notebook. It tunes both models on training data, selects the
candidate using validation MAE, and reports the test results only after that
selection. The code is intentionally short and uses scikit-learn functions for
loading, tuning, fitting, metrics, and feature importance.

**Required files:** keep `training.csv` (or `train.csv`), `validation.csv`, and
`test.csv` in this folder. The exported files are already imputed and one-hot
encoded by the EDA notebook. Do not fit another imputer or encoder here.

## 1. Imports and settings

The team EDA uses a random 60/20/20 split. These model results therefore
describe a random-holdout evaluation, not a later-in-time forecasting test.
`SEED` makes the model fitting and tuning reproducible.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

SEED = 505
TARGET = "delivery_duration"
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)


## 2. Load the prepared partitions

The helper accepts either common training-file name. It stops with a clear error
if the training partition is missing: a validation or test set cannot be used to
train the models. The checks confirm equal feature columns, no missing values,
and a positive duration target.

In [2]:
def load_split(*names):
    # Load the first available CSV name from the project folder.
    for name in names:
        path = Path(name)
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(f"Missing one of: {', '.join(names)}")


train = load_split("training.csv", "train.csv")
validation = load_split("validation.csv")
test = load_split("test.csv")
partitions = {"Train": train, "Validation": validation, "Test": test}

for name, frame in partitions.items():
    print(f"{name}: {frame.shape[0]:,} rows, {frame.shape[1]} columns")
    assert frame[TARGET].gt(0).all(), f"{name} has an invalid target"
    assert frame.isna().sum().sum() == 0, f"{name} has missing values"

features = train.columns.drop(TARGET)
assert features.equals(validation.columns.drop(TARGET))
assert features.equals(test.columns.drop(TARGET))


Train: 108,331 rows, 107 columns
Validation: 36,111 rows, 107 columns
Test: 36,111 rows, 107 columns


## 3. Create familiar train, validation, and test variables

The target is delivery duration in minutes. All remaining columns are numeric,
preprocessed order-time features from the EDA notebook. We retain the supplied
splits rather than splitting them again.

In [3]:
X_train, y_train = train[features], train[TARGET]
X_valid, y_valid = validation[features], validation[TARGET]
X_test, y_test = test[features], test[TARGET]

display(pd.DataFrame({
    name: frame[TARGET].describe()[["mean", "50%", "std", "min", "max"]]
    for name, frame in partitions.items()
}).round(2))


,Train,Validation,Test
mean,47.74,47.55,47.46
50%,44.42,44.33,44.28
std,18.57,18.28,18.28
min,1.68,3.72,5.87
max,297.70,262.78,282.23


## 4. Define and tune Models 3 and 4

Random Forest averages many decision trees. Histogram Gradient Boosting adds
small trees sequentially and uses absolute-error loss, which is less sensitive
to large delivery-duration errors than squared-error loss. `GridSearchCV` uses
three folds from training data only. The small grids keep the process readable
and reproducible instead of searching hundreds of combinations.

In [4]:
model_specs = {
    "Random Forest": (
        RandomForestRegressor(
            n_estimators=150, max_features=0.8, random_state=SEED, n_jobs=1,
        ),
        {"max_depth": [12, 18], "min_samples_leaf": [10, 25]},
    ),
    "Gradient Boosting": (
        HistGradientBoostingRegressor(
            loss="absolute_error", learning_rate=0.08, max_iter=150,
            early_stopping=False, random_state=SEED,
        ),
        {"max_leaf_nodes": [15, 31], "l2_regularization": [0.0, 1.0]},
    ),
}


def metrics(actual, predicted):
    # Use one definition of the regression metrics for every model.
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": root_mean_squared_error(actual, predicted),
        "R-squared": r2_score(actual, predicted),
        "Bias": (predicted - actual).mean(),
    }


## 5. Fit on training data and select with validation data

The loop replaces nearly identical model-specific code. It tunes each model
inside training, predicts the untouched validation partition, and stores its
best fitted estimator. The median baseline shows whether either model improves
on a simple constant ETA. The model with the lowest validation MAE is selected
before viewing test results.

In [5]:
fitted_models = {}
validation_rows = []

baseline = DummyRegressor(strategy="median").fit(X_train, y_train)
validation_rows.append({
    "Model": "Median baseline",
    "Parameters": "median training duration",
    **metrics(y_valid, baseline.predict(X_valid)),
})

for name, (model, grid) in model_specs.items():
    search = GridSearchCV(
        model, grid, cv=3, scoring="neg_mean_absolute_error", n_jobs=1,
    ).fit(X_train, y_train)
    fitted_models[name] = search.best_estimator_
    validation_rows.append({
        "Model": name,
        "Parameters": str(search.best_params_),
        **metrics(y_valid, fitted_models[name].predict(X_valid)),
    })

validation_results = pd.DataFrame(validation_rows).set_index("Model")
selected_name = validation_results["MAE"].idxmin()
display(validation_results.round(3))
print("Validation-selected model:", selected_name)


,Parameters,MAE,RMSE,R-squared,Bias
Model,,,,,
Median baseline,median training duration,13.330,18.550,-0.029,-3.131
Random Forest,"{'max_depth': 18, 'min_samples_leaf': 10}",10.872,14.999,0.327,0.185
Gradient Boosting,"{'l2_regularization': 1.0, 'max_leaf_nodes': 31}",10.399,14.988,0.328,-2.418


Validation-selected model: Gradient Boosting


## 6. Explain the validation models

Permutation importance measures how much validation MAE worsens when one feature
is shuffled. It describes predictive usefulness in this model; it does not prove
that changing the feature causes faster or slower deliveries.

In [6]:
importance_rows = []
importance_sample = validation.sample(min(3000, len(validation)), random_state=SEED)

for name, model in fitted_models.items():
    result = permutation_importance(
        model, importance_sample[features], importance_sample[TARGET],
        scoring="neg_mean_absolute_error", n_repeats=3, random_state=SEED,
        n_jobs=1,
    )
    importance_rows.append(pd.DataFrame({
        "Feature": features,
        "MAE increase": result.importances_mean,
        "Model": name,
    }))

importance = pd.concat(importance_rows, ignore_index=True)
display(importance.groupby("Model").apply(
    lambda frame: frame.nlargest(5, "MAE increase"), include_groups=False,
).round(3))


Feature  \
Model                                                                 
Gradient Boosting 116                      total_outstanding_orders   
                  114                         total_onshift_dashers   
                  118  estimated_store_to_consumer_driving_duration   
                  106                                    order_hour   
                  109                                      subtotal   
Random Forest     10                       total_outstanding_orders   
                  8                           total_onshift_dashers   
                  0                                      order_hour   
                  12   estimated_store_to_consumer_driving_duration   
                  3                                        subtotal   

                       MAE increase  
Model                                
Gradient Boosting 116         5.966  
                  114         4.710  
                  118         1.115  
                  106         0.861  
                  109         0.389  
Random Forest     10          3.292  
                  8           2.375  
                  0           1.377  
                  12          1.082  
                  3           0.453

## 7. Refit selected settings and evaluate the test partition

The selected settings are refit using train plus validation data. Test results
are reported for both models, but the choice remains the validation-selected
model. Do not choose a new winner after seeing the test table.

In [7]:
X_train_valid = pd.concat([X_train, X_valid])
y_train_valid = pd.concat([y_train, y_valid])
final_models = {}
test_rows = []

for name, model in fitted_models.items():
    final_models[name] = clone(model).fit(X_train_valid, y_train_valid)
    test_rows.append({
        "Model": name,
        **metrics(y_test, final_models[name].predict(X_test)),
    })

final_baseline = DummyRegressor(strategy="median").fit(X_train_valid, y_train_valid)
test_rows.append({
    "Model": "Median baseline",
    **metrics(y_test, final_baseline.predict(X_test)),
})
test_results = pd.DataFrame(test_rows).set_index("Model")
display(test_results.sort_values("MAE").round(3))


,MAE,RMSE,R-squared,Bias
Model,,,,
Gradient Boosting,10.418,15.102,0.317,-2.476
Random Forest,10.838,15.084,0.319,0.185
Median baseline,13.291,18.533,-0.028,-3.058


## 8. Findings and reproducible exports

The data were randomly split and preprocessed
before export. The results estimate performance on similar historical orders;
they do not establish a causal operational change or confirm performance on a
later independent period.

In [8]:
selected_test_mae = test_results.loc[selected_name, "MAE"]
baseline_test_mae = test_results.loc["Median baseline", "MAE"]
reduction = 100 * (1 - selected_test_mae / baseline_test_mae)
top_features = importance.loc[
    importance["Model"].eq(selected_name)
].nlargest(3, "MAE increase")["Feature"].tolist()

print(f"Selected model: {selected_name}")
print(f"Validation MAE: {validation_results.loc[selected_name, 'MAE']:.2f} minutes")
print(f"Test MAE: {selected_test_mae:.2f} minutes")
print(f"Median-baseline test MAE: {baseline_test_mae:.2f} minutes")
print(f"MAE reduction versus baseline: {reduction:.1f}%")
print("Top validation features:", ", ".join(top_features))

validation_results.to_csv(RESULTS / "models_3_4_validation_results.csv")
test_results.to_csv(RESULTS / "models_3_4_test_results.csv")
importance.to_csv(RESULTS / "models_3_4_importance.csv", index=False)


Selected model: Gradient Boosting
Validation MAE: 10.40 minutes
Test MAE: 10.42 minutes
Median-baseline test MAE: 13.29 minutes
MAE reduction versus baseline: 21.6%
Top validation features: total_outstanding_orders, total_onshift_dashers, estimated_store_to_consumer_driving_duration
